# Data Visualization and Pipeline Audit

Run this notebook after Gold data has been built.

It displays canonical inputs, intermediate transformations, and final tables before and after each processing stage. Change the preview controls when more rows are needed; the notebook does not overwrite Drive data.


## 1. Colab setup

Cell tiếp theo thực hiện ba việc:

- Mount Google Drive để đọc `MyDrive/nba-scout-assistant/data`.
- Clone repository nếu chưa tồn tại; nếu đã clone thì pull code mới nhất.
- Cài các package chỉ cần cho việc đọc bảng.

Repository cung cấp transformation code; Drive cung cấp raw/silver/gold data. Hai phần được giữ riêng để notebook không phụ thuộc vào data được commit trong Git.

In [ ]:
!pip -q install pyarrow openpyxl

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/kdnehihi/nba-scout-assistant.git'
REPO_DIR = Path('/content/nba-scout-assistant')

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repository:', REPO_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 2. Data root and display configuration

`DATA_DIR` phải là folder chứa trực tiếp các layer `raw`, `silver`, `gold` và có thể có `bronze`. Mặc định notebook dùng đúng địa chỉ Drive của dự án.

- `PREVIEW_ROWS` điều khiển số row mẫu trong mỗi bảng.
- Tất cả columns luôn được hiển thị; schema table liệt kê đầy đủ từng column.
- Đặt `DISPLAY_ALL_ROWS = True` chỉ khi muốn render toàn bộ dataframe. Với game logs hàng trăm nghìn dòng, Colab có thể chậm hoặc hết RAM.

In [ ]:
from IPython.display import display, Markdown
from google.colab import data_table
import numpy as np
import pandas as pd

from src.dataset.loaders import (
    list_tabular_files,
    load_player_game_logs,
    load_player_season_salaries,
    load_player_season_stats,
    load_players,
    load_salary_cap,
    resolve_data_paths,
)
from src.dataset.season_coverage import (
    filter_to_modeling_seasons,
    modeling_seasons_through_latest_complete,
    summarize_game_log_season_coverage,
)
from src.dataset.features_role import build_role_features
from src.dataset.features_performance import add_rolling_player_features, build_performance_training
from src.dataset.features_compensation import build_player_salary_history
from src.dataset.features_long_term import (
    add_future_horizon_targets,
    add_lagged_season_features,
    build_long_term_inference,
    build_long_term_training,
    build_player_season_summary,
    build_recent_game_anchor_features,
)

DATA_DIR = Path('/content/drive/MyDrive/nba-scout-assistant/data')
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Drive data folder was not found: {DATA_DIR}. '
        'Expected raw/, silver/, and gold/ directly below this path.'
    )

paths = resolve_data_paths(DATA_DIR)
PREVIEW_ROWS = 20
DISPLAY_ALL_ROWS = False

data_table.enable_dataframe_formatter()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 240)

print('DATA_DIR:', paths.data_dir)
print('raw:', paths.raw_dir.exists(), paths.raw_dir)
print('silver:', paths.silver_dir.exists(), paths.silver_dir)
print('gold:', paths.gold_dir.exists(), paths.gold_dir)

## 3. Visualization helpers

Mỗi stage được mô tả bằng hai bảng:

- **Data preview:** tất cả columns và một số rows đại diện.
- **Schema:** dtype, số missing, tỷ lệ missing, số unique và một giá trị mẫu cho từng column.

`compare_stages` không giả định hai bảng có cùng grain. Nó chỉ cho biết shape thay đổi ra sao và columns nào được thêm/bỏ. Grain và key của từng bảng được giải thích trong markdown ngay trước stage tương ứng.

In [ ]:
def schema_table(df: pd.DataFrame) -> pd.DataFrame:
    sample_values = []
    for column in df.columns:
        non_null = df[column].dropna()
        sample_values.append(non_null.iloc[0] if len(non_null) else None)
    return pd.DataFrame({
        'column': df.columns,
        'dtype': [str(dtype) for dtype in df.dtypes],
        'missing_count': df.isna().sum().to_numpy(),
        'missing_pct': (df.isna().mean().mul(100).round(2)).to_numpy(),
        'n_unique': [df[column].nunique(dropna=True) for column in df.columns],
        'sample_value': sample_values,
    })


def show_stage(title: str, df: pd.DataFrame, rows: int = PREVIEW_ROWS) -> None:
    display(Markdown(f'### {title}'))
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    preview = df if DISPLAY_ALL_ROWS else df.head(rows)
    display(preview)
    display(Markdown('**Complete schema and missingness**'))
    display(schema_table(df))


def compare_stages(before: pd.DataFrame, after: pd.DataFrame) -> pd.DataFrame:
    before_columns = set(before.columns)
    after_columns = set(after.columns)
    return pd.DataFrame({
        'measure': ['rows', 'columns', 'missing_cells', 'duplicate_rows', 'columns_added', 'columns_removed'],
        'before': [
            len(before),
            before.shape[1],
            int(before.isna().sum().sum()),
            int(before.duplicated().sum()),
            None,
            None,
        ],
        'after': [
            len(after),
            after.shape[1],
            int(after.isna().sum().sum()),
            int(after.duplicated().sum()),
            sorted(after_columns - before_columns),
            sorted(before_columns - after_columns),
        ],
    })


def show_before_after(
    before_name: str,
    before: pd.DataFrame,
    after_name: str,
    after: pd.DataFrame,
    rows: int = PREVIEW_ROWS,
) -> None:
    show_stage(f'BEFORE - {before_name}', before, rows=rows)
    display(Markdown('### Structural change summary'))
    display(compare_stages(before, after))
    show_stage(f'AFTER - {after_name}', after, rows=rows)

## 4. Source inventory and canonical inputs

### Layer meanings

| Layer | Meaning in this project |
|---|---|
| `raw` | Source-level or canonical source tables: player bio, advanced season stats, cap history and downloaded snapshots. |
| `bronze` | Downloaded/staged source files retained close to their original form. |
| `silver` | Normalized reusable tables. Game logs and salary rows have stable names/types here. |
| `gold` | Task-ready feature tables, inference anchors and deterministic scouting outputs. |

### Canonical inputs loaded by `src/dataset/loaders.py`

- `load_players`: player identity and bio. Grain: one row per player.
- `load_player_game_logs`: canonical game observations. Grain: one player in one game.
- `load_player_season_stats`: advanced/rate statistics. Grain: player-season-team.
- `load_player_season_salaries`: normalized salary history. Grain: player-season salary row.
- `load_salary_cap`: league cap context. Grain: one row per season.

Loaders validate required columns but intentionally do not engineer model features.

In [ ]:
inventory_rows = []
for layer_name, layer_path in [
    ('bronze', paths.bronze_dir),
    ('raw', paths.raw_dir),
    ('silver', paths.silver_dir),
    ('gold', paths.gold_dir),
]:
    if not layer_path.exists():
        continue
    for file_path in list_tabular_files(layer_path):
        inventory_rows.append({
            'layer': layer_name,
            'relative_path': str(file_path.relative_to(paths.data_dir)),
            'extension': file_path.suffix.lower(),
            'size_mb': round(file_path.stat().st_size / 1024**2, 3),
        })

inventory = pd.DataFrame(inventory_rows).sort_values(['layer', 'relative_path']).reset_index(drop=True)
display(inventory)

players = load_players(paths)
game_logs_raw = load_player_game_logs(paths)
season_stats_raw = load_player_season_stats(paths)
salaries_raw = load_player_season_salaries(paths)
salary_cap_raw = load_salary_cap(paths)

for name, dataframe in {
    'players (raw canonical input)': players,
    'player_game_logs (silver canonical input)': game_logs_raw,
    'player_season_stats (raw canonical input)': season_stats_raw,
    'player_season_salaries (silver canonical input)': salaries_raw,
    'salary_cap_by_season (raw canonical input)': salary_cap_raw,
}.items():
    show_stage(name, dataframe)

## 5. Complete-season coverage filter

**Code owner:** `src/dataset/season_coverage.py`

**Functions:**

- `summarize_game_log_season_coverage` aggregates rows, unique games, players, teams and date range by season.
- A season is considered modeling-complete when it has at least **1,000 unique games** and **30 teams**.
- `modeling_seasons_through_latest_complete` identifies the latest complete season and retains all earlier seasons.
- `filter_to_modeling_seasons` applies the approved season set to game logs and advanced season stats.

This step prevents partial recent seasons from becoming misleading training data. It is a row filter, not missing-value imputation.

In [ ]:
season_coverage = summarize_game_log_season_coverage(game_logs_raw)
modeling_seasons = modeling_seasons_through_latest_complete(game_logs_raw)
game_logs = filter_to_modeling_seasons(game_logs_raw, modeling_seasons)
season_stats = filter_to_modeling_seasons(season_stats_raw, modeling_seasons)

display(Markdown('### Season-level coverage decision'))
display(season_coverage)
print('Approved modeling seasons:', sorted(modeling_seasons))

show_before_after(
    'all canonical game-log seasons', game_logs_raw,
    'game logs through latest complete season', game_logs,
)
show_before_after(
    'all canonical advanced-stat seasons', season_stats_raw,
    'advanced stats through latest complete season', season_stats,
)

## 6. Role and recommendation feature table

**Code owner:** `src/dataset/features_role.py`

**Builder:** `build_role_features(players, season_stats, game_logs)`

### Transformation design

1. Starts from advanced `season_stats` at player-season-team grain.
2. Calls `build_player_season_summary` to derive production rates from complete game logs; these are preferred for points/assists/rebounds per 100 possessions.
3. Merges bio columns (`birth_date`, `position`, `height`, `weight`) by `player_id`.
4. Converts mixed percent scales to ratios using `percent_to_ratio` from `src/dataset/cleaning.py`. Example: `55.0 -> 0.55`, while `0.55` remains `0.55`.
5. Adds `<column>_was_missing` flags before filling numeric missing values with column medians and categorical values with `UNK`. The flags preserve knowledge that a value was originally absent.
6. Adds interpretable dimensions such as `scoring_creation`, `playmaking`, `shooting`, `rebounding`, defensive dimensions and `two_way_impact`.
7. Removes duplicate `(player_id, season, team_id)` rows.

**Final use:** player recommendation and player-detail scouting profiles. The role dimensions are deterministic summaries; raw rate stats remain available for distance calculations and audit.

In [ ]:
role_features = build_role_features(players, season_stats, game_logs=game_logs)
show_before_after(
    'advanced player-season stats', season_stats,
    'player role features', role_features,
)

role_added_columns = sorted(set(role_features.columns) - set(season_stats.columns))
display(Markdown('### Columns added by bio merge, game-log production, missing flags and role dimensions'))
display(pd.DataFrame({'added_column': role_added_columns}))

## 7. Short-term performance training table

**Code owner:** `src/dataset/features_performance.py`

### Stage A - `add_rolling_player_features`

- Sorts by `(player_id, season, as_of_date, game_id)` so every rolling window follows time within one player-season.
- Creates last-5, last-10 and expanding season averages for PTS, AST, REB and MIN.
- Creates form deltas such as `pts_last_5_minus_season_avg`.
- Creates supervised targets `target_next_5_*_avg`. At row `t`, the target uses games `t+1` through `t+5`; the current game is not included.

### Stage B - `build_performance_training`

- Maps each season to `train`, `validation`, `test` or `ignore` using `src/dataset/splits.py`.
- Drops rows without complete rolling features or next-five-game targets. This removes early-season rows without enough history and late-season rows without five future games.
- Keeps only train/validation/test rows.

**Grain after processing:** one player-game anchor with information available at that date and labels observed over the following five games.

In [ ]:
performance_with_rolling = add_rolling_player_features(game_logs)
performance_training = build_performance_training(game_logs)

eligible_counts = game_logs.groupby(['player_id', 'season']).size()
sample_player_id, sample_season = eligible_counts[eligible_counts.ge(20)].index[0]
sample_before = game_logs[
    game_logs['player_id'].eq(sample_player_id)
    & game_logs['season'].eq(sample_season)
].sort_values(['game_date', 'game_id'])
sample_after = performance_with_rolling[
    performance_with_rolling['player_id'].eq(sample_player_id)
    & performance_with_rolling['season'].eq(sample_season)
].sort_values(['as_of_date', 'game_id'])

print('Timeline example:', sample_player_id, sample_season)
show_before_after(
    'one player-season game timeline', sample_before,
    'same timeline with rolling features and future targets', sample_after,
    rows=25,
)
show_before_after(
    'all complete-season game logs', game_logs,
    'final short-term training rows', performance_training,
)
display(Markdown('### Split counts'))
display(performance_training['split'].value_counts(dropna=False).rename_axis('split').reset_index(name='rows'))

## 8. Salary and contract context table

**Code owner:** `src/dataset/features_compensation.py`

**Builder:** `build_player_salary_history(salaries, salary_cap, players)`

This table is reporting context for candidate detail pages; the current product does not forecast salary.

### Transformation design

1. Parses `salary_usd` into numeric USD.
2. Canonicalizes team abbreviations, for example historical/source aliases to one team ID.
3. Creates a normalized `player_name_key` for fallback identity joins.
4. Left-joins salary cap by `season_label` and computes `salary_cap_share = salary_usd / salary_cap_usd`. This expresses salary relative to the league cap in that season.
5. If salary data lacks `player_id`, joins it from player bio through the normalized name key.
6. Keeps reporting columns, drops rows missing player name, season or salary, then sorts chronologically by player.

The optional contract-event file is loaded separately by application code when available; it is not required to construct salary history.

In [ ]:
salary_history = build_player_salary_history(salaries_raw, salary_cap_raw, players)

show_stage('BEFORE - normalized salary rows', salaries_raw)
show_stage('BEFORE - salary cap lookup', salary_cap_raw)
display(Markdown('### Structural change from salary rows to reporting history'))
display(compare_stages(salaries_raw, salary_history))
show_stage('AFTER - player salary history with cap context', salary_history)

## 9. Long-term forecasting tables

**Code owner:** `src/dataset/features_long_term.py`

Long-term processing changes the grain from player-game to player-season anchor through several explicit stages:

| Stage | Function | Input -> output | Main changes |
|---|---|---|---|
| Season aggregation | `build_player_season_summary` | player-game -> player-season | Totals, MPG, availability, per-36, estimated per-100, bio, age, career totals. |
| Recent form | `build_recent_game_anchor_features` | last 20 games -> player-season | Recent per-36 rates and linear trends. |
| Career history | `add_lagged_season_features` | ordered seasons -> anchor row | Current and previous four seasons become `*_lag_0` to `*_lag_3`; recent 3-year slopes are added. |
| Future labels | `add_future_horizon_targets` | anchor + season lookup -> labeled anchor | Adds observed H1/H2/H3 active, per-36, per-100 and per-game values. |
| Training | `build_long_term_training` | all stages -> labeled Gold | Requires complete H1-H3 active labels, assigns temporal split and excludes ignored anchors. |
| Inference | `build_long_term_inference` | all feature stages -> unlabeled Gold | Keeps latest available anchors because future labels are not required for prediction. |

### Important target semantics

When a future season is inside observed data but the player has no row, `active_hN = 0`. When the future season is beyond observed coverage, the label remains missing rather than pretending the player was inactive. This distinction prevents unknown future outcomes from being converted into false negatives.

In [ ]:
season_summary = build_player_season_summary(game_logs, players, season_stats)
recent_features = build_recent_game_anchor_features(game_logs)
lagged_features = add_lagged_season_features(season_summary)
labeled_anchors = add_future_horizon_targets(lagged_features, season_summary)
long_term_training = build_long_term_training(game_logs, players, season_stats)
long_term_inference = build_long_term_inference(game_logs, players, season_stats)

show_before_after(
    'complete-season player game logs', game_logs,
    'player-season summary', season_summary,
)
show_stage('INTERMEDIATE - recent 20-game anchor features', recent_features)
show_before_after(
    'player-season summary', season_summary,
    'lagged player-season anchor features', lagged_features,
)
show_before_after(
    'unlabeled lagged anchors', lagged_features,
    'anchors with H1/H2/H3 observed targets', labeled_anchors,
)
show_stage('AFTER - final long-term training table', long_term_training)
show_stage('AFTER - final long-term inference table', long_term_inference)

display(Markdown('### Long-term training split counts'))
display(long_term_training['split'].value_counts(dropna=False).rename_axis('split').reset_index(name='rows'))

## 10. Final in-memory Gold catalog

`src/dataset/pipeline.py::build_all_gold_datasets` orchestrates the same order used above:

```text
load canonical inputs
-> summarize and filter complete seasons
-> build role features
-> build short-term performance training
-> build salary history context
-> build long-term training and inference anchors
-> persist each dataframe under data/gold/*.parquet
```

The catalog below summarizes the outputs reconstructed in memory. It does not overwrite Drive. Use the selector in the following cell to inspect any table with all columns and complete schema metadata.

In [ ]:
gold_in_memory = {
    'player_role_features_clean': role_features,
    'performance_training_clean': performance_training,
    'player_salary_history_clean': salary_history,
    'long_term_player_forecast_training': long_term_training,
    'long_term_player_forecast_inference': long_term_inference,
    'season_coverage': season_coverage,
}

catalog = pd.DataFrame([
    {
        'dataset': name,
        'rows': len(df),
        'columns': df.shape[1],
        'missing_cells': int(df.isna().sum().sum()),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        'materialized_path': str(paths.gold_dir / f'{name}.parquet'),
        'exists_on_drive': (paths.gold_dir / f'{name}.parquet').exists(),
    }
    for name, df in gold_in_memory.items()
])
display(catalog)

# Change this value to inspect another complete Gold table.
SELECTED_DATASET = 'performance_training_clean'
show_stage(f'SELECTED GOLD - {SELECTED_DATASET}', gold_in_memory[SELECTED_DATASET], rows=50)

## 11. Source-to-output lineage summary

| Output | Primary inputs | Transformation file | Key grain |
|---|---|---|---|
| `player_role_features_clean.parquet` | players + season stats + game logs | `features_role.py` | player-season-team |
| `performance_training_clean.parquet` | game logs | `features_performance.py` | player-game anchor |
| `player_salary_history_clean.parquet` | salaries + salary cap + players | `features_compensation.py` | player-season salary row |
| `long_term_player_forecast_training.parquet` | game logs + players + season stats | `features_long_term.py` | labeled player-season anchor |
| `long_term_player_forecast_inference.parquet` | game logs + players + season stats | `features_long_term.py` | inference player-season anchor |
| `season_coverage.parquet` | game logs | `season_coverage.py` | season |

### Review checklist

- Verify each canonical input path before interpreting missing columns.
- Confirm the latest complete season in the coverage table.
- Inspect missing flags in role features before trusting imputed values.
- Confirm short-term rows have both sufficient history and five future games.
- Confirm long-term training anchors have observed H1-H3 labels, while inference anchors may extend to the latest complete season.
- Treat salary and optional contract data as historical context, not a salary forecast.

For implementation details beyond the tables, see `docs/data_pipeline.md`.